In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from __future__ import annotations

from PIL import Image
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas
import pytesseract
from pytesseract import Output

from tqdm import tqdm

# docx2pdf is optional (requires Word/pywin32 on Windows).
try:
    from docx2pdf import convert as docx2pdf_convert
except Exception:
    docx2pdf_convert = None

c:\Users\sandi\Desktop\ML Working Folder\talentlens_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = Path.cwd()
print(f"The current working directory is : {current_dir}")

project_root = current_dir.parent
print(f"The current working directory is : {project_root}")

original_resume_folder_path = project_root / "data" / "original"
print(f"The current working directory is : {original_resume_folder_path}")

processed_resume_folder_path = project_root / "data" / "processed"
print(f"The current working directory is : {processed_resume_folder_path}")

The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\talentlens_ai\notebooks
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\talentlens_ai
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\talentlens_ai\data\original
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\talentlens_ai\data\processed


In [35]:
# ----------------------------
# Config
# ----------------------------

ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".doc", ".docx"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

# If you want to keep .doc/.docx rather than deleting them when OCR isn't handled for them:
# set to True to delete .doc/.docx that aren't converted.
DELETE_DOCS_IF_NOT_CONVERTED = False

# ----------------------------
# Helpers
# ----------------------------

def ocr_and_embed_text_to_pdf(img: Image.Image, dst: Path) -> None:
    """Embed an image into a single-page PDF and write OCR text as real PDF text."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    page_size = A4
    w_pt, h_pt = page_size

    img_rgb = img.convert("RGB")
    img_w, img_h = img_rgb.size

    # Fit image to A4 while preserving aspect ratio
    scale = min(w_pt / img_w, h_pt / img_h)
    draw_w = img_w * scale
    draw_h = img_h * scale

    x0 = (w_pt - draw_w) / 2
    y0 = (h_pt - draw_h) / 2

    c = canvas.Canvas(str(dst), pagesize=page_size)
    img_reader = ImageReader(img_rgb)

    # Draw original image
    c.drawImage(
        img_reader,
        x0,
        y0,
        width=draw_w,
        height=draw_h,
        preserveAspectRatio=True,
        mask="auto",
    )

    # OCR with bounding boxes
    data = pytesseract.image_to_data(img_rgb, output_type=Output.DICT)

    # Write OCR text as actual PDF text.
    n = len(data["text"])
    for i in range(n):
        word = (data["text"][i] or "").strip()
        if not word:
            continue

        left = data["left"][i]
        top = data["top"][i]
        width = data["width"][i]
        height = data["height"][i]

        # Map word bbox from image coords (top-left origin) to PDF coords (bottom-left origin)
        x_pdf = x0 + (left / img_w) * draw_w
        y_pdf = y0 + (1 - (top / img_h)) * draw_h

        # Font size heuristic from OCR box height
        font_size = max(6, (height / img_h) * h_pt * 0.9)

        # Light gray fill to reduce visual impact but keep text extractable
        c.setFillGray(0.2)
        c.setFont("Helvetica", font_size)

        c.drawString(x_pdf, y_pdf, word)

    c.showPage()
    c.save()


def should_skip_or_delete(src: Path) -> bool:
    ext = src.suffix.lower()
    if ext in ALLOWED_EXTS:
        return False
    # Delete anything not allowed
    src.unlink(missing_ok=True)
    return True


# ----------------------------
# Main
# ----------------------------
if not original_resume_folder_path.exists():
    raise FileNotFoundError(f"Source root not found: {original_resume_folder_path.resolve()}")

processed_resume_folder_path.mkdir(parents=True, exist_ok=True)

all_files = [p for p in original_resume_folder_path.rglob("*") if p.is_file()]

with tqdm(all_files, desc="OCR->PDF conversion", unit="file") as pbar:
    for src in pbar:
        ext = src.suffix.lower()

        if should_skip_or_delete(src):
            pbar.set_postfix_str(f"deleted: {src.name}")
            continue

        rel = src.relative_to(original_resume_folder_path)
        dst = (processed_resume_folder_path / rel).with_suffix(".pdf")

        # Skip if already exists
        if dst.exists() and dst.stat().st_size > 0:
            pbar.set_postfix_str(f"skipped: {src.parent.name} | {src.name}")
            continue

        pbar.set_postfix_str(f"{src.parent.name} | {src.name}")

        # Images -> OCR-embedded PDF
        if ext in IMAGE_EXTS:
            img = Image.open(src)
            ocr_and_embed_text_to_pdf(img, dst)
            continue

        # DOC/DOCX: OCR conversion not implemented here.
        # Keep or delete according to DELETE_DOCS_IF_NOT_CONVERTED.
        if ext in {".docx", ".doc"}:
            if DELETE_DOCS_IF_NOT_CONVERTED:
                src.unlink(missing_ok=True)
            continue

print("OCR->PDF conversion complete. Output at:", processed_resume_folder_path.resolve())

OCR->PDF conversion:   0%|          | 0/907 [00:00<?, ?file/s, BusinessAnalyst | 01888170110d1ccf.png]


TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.